# Latent space generation

In [1]:
import os
# Set environment variables to disable multithreading
# as users will probably want to set the number of cores
# to the max of their computer.
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

In [2]:
import glob
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import seaborn as sns

from anomaly.constants import GALAXY_LINES
from anomaly.utils import specobjid_to_idx
from anomaly.utils import VelocityFilter
from autoencoders.ae import AutoEncoder

from sdss.metadata import MetaData

meta = MetaData()

# Custom Functions

# Config

## Directories

In [34]:
phd_dir = "/home/elom/phd"
thesis_dir = f"{phd_dir}/thesis"
data_dir = f"{phd_dir}/code"
spectra_dir = f"{data_dir}/spectra"
models_dir = f"{data_dir}/models"
latent_dir = f"{data_dir}/latent"
bins_ids = [f'bin_{i:02d}' for i in range(4)] 

## Data

In [16]:
wave = np.load(f"{spectra_dir}/wave_spectra_imputed.npy")
wave_nm = wave*0.1
n_wave = wave.shape

spectra = np.load(
    f"{spectra_dir}/spectra_imputed.npy",
    mmap_mode="r"
)

final_meta_df = pd.read_csv(
    f"{spectra_dir}/final_spec_n_z_warning_drop.csv.gz",
    index_col="specobjid",
)

idx_id_spec = np.load(
    f"{spectra_dir}/ids_imputing.npy",
    mmap_mode='r'
)


## Spec per bin

In [22]:
spectra_bin_dict = {}

for bin_id in bins_ids:

    idxs = np.load(
        f"{spectra_dir}/{bin_id}/{bin_id}_index_specobjid.npy"
    )[:, 0]

    spectra_bin_dict[bin_id] = spectra[idxs]


In [23]:
spectra_bin_dict[bin_id].shape

(181850, 3773)

# bin_03

In [25]:

'bin_03' in winner_models_paths[0]

True

In [ ]:
winner_models_paths = glob.glob(
    f"{models_dir}/bin_*/winn*"
)

models_dict = {}
latent_dict = {}

for bin_id in bins_ids:

    model_path = glob.glob(f"{models_dir}/{bin_id}/winn*")[0]

    model = AutoEncoder(reload=True, reload_from=model_path)
    models_dict[bin_id] = model
    
    print(f"Encode {bin_id} with {model_path}")
    
    latent_dict[bin_id] = model.encode(spectra_bin_dict[bin_id])
    
    save_to = f"{latent_dir}/{bin_id}" 
    os.makedirs(save_to, exist_ok=True)

    np.save(
        f"{save_to}/latent_{bin_id}.npy",
        latent_dict[bin_id]
    )




Encode bin_00 with /home/elom/phd/code/models/bin_00/winner_0051
Encode bin_01 with /home/elom/phd/code/models/bin_01/winner_0063
Encode bin_02 with /home/elom/phd/code/models/bin_02/winner_0021
Encode bin_03 with /home/elom/phd/code/models/bin_03/winner
